In [2]:
import sqlite3
import pandas as pd

In [3]:
con = sqlite3.connect('ecommerce-sales.db')

In [4]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE TYPE = 'table'",con)

In [5]:
tables['name']


0       categories
1        customers
2           orders
3      order_items
4         payments
5         products
6       sales_info
7    customer_info
8    products_info
Name: name, dtype: object

In [6]:
for table in tables['name']:
    print(table)

categories
customers
orders
order_items
payments
products
sales_info
customer_info
products_info


## **------------------Shows Top 10 records of each table----------------------------**

In [ ]:
for table in tables['name']:
    print(f"-----------------------------------{table.upper()}---------------------------------")
    display("Count of Records",pd.read_sql(f"SELECT COUNT(*) as Count FROM {table}",con)['Count'].values[0])
    display(pd.read_sql(f"SELECT * FROM {table}",con))

# **------------------Shows Datas in their respective Tables----------------------------**

In [ ]:
categories = pd.read_sql("SELECT * FROM categories",con)
categories

In [ ]:
products = pd.read_sql("SELECT * FROM products",con)
products

In [ ]:
customers = pd.read_sql("SELECT * FROM customers",con)
customers

In [ ]:
orders = pd.read_sql("SELECT * FROM orders",con)
orders

In [ ]:
order_items = pd.read_sql("SELECT * FROM order_items",con)
order_items

In [ ]:
payments = pd.read_sql("SELECT * FROM payments",con)
payments

## **------------------Shows NULLs records of each table----------------------------**

In [ ]:

for table in tables['name']:
    print(f"-----------------------------------{table.upper()}---------------------------------")
    display(pd.read_sql(f"SELECT * FROM {table}",con).isna().sum())

## **------------------Filling NULLs records In Table 'customers'----------------------------**

In [ ]:
customers['email'] = customers['email'].fillna('n/a')

In [ ]:
customers['email'].isna().sum()

In [ ]:
customers.info()

In [ ]:
customers['date_of_birth'] = customers['date_of_birth'].fillna('0000-00-00')

In [ ]:
customers['date_of_birth'].isna().sum()

In [ ]:

for table in tables['name']:
    print(f"-----------------------------------{table.upper()}---------------------------------")
    display(pd.read_sql(f"SELECT * FROM {table}",con).duplicated().sum())

In [ ]:
orders.duplicated().sum()

In [ ]:
customers

# **------------------------------- Customer Table -----------------------------**


## **------------------ Created cleaned Tabel 'customer' ----------------------------**

In [ ]:
customers = pd.read_sql(
    """
    SELECT
        c.customer_id,
        CONCAT(first_name,' ', last_name) AS customer_name,
        c.email,
        c.phone,
        c.gender,
        c.date_of_birth,
        c.city,
        c.state,
        c.country,
        c.signup_date,
        CASE
            WHEN sum(oi.line_total) > 200000 THEN 'HIGH'
            WHEN sum(oi.line_total) > 100000 THEN 'MEDIUN'
            ELSE 'LOW'
        END AS cuatomer_segment,
        c.is_subscribed_newsletter
        
        
    FROM customers c
    LEFT JOIN orders o
    On c.customer_id = o.customer_id
    JOIN order_items oi
    ON o.order_id = oi.order_id
    RIGHT JOIN products p
    ON p.product_id = oi.product_id
    JOIN payments ps
    ON ps.order_id = o.order_id 
    GROUP BY c.customer_id 
    """,con
)



Connected successfully!


In [ ]:
customers["phone"] = (
    customers["phone"]
    .astype("string")
    .str.replace(r"\D", "", regex=True)  # Keep only numbers
    .str.replace(r"^91", "", regex=True) # Remove country code 91
)

In [18]:
print(type(con))

<class 'pyodbc.Connection'>


In [ ]:
customers

In [ ]:
for table in tables['name']:
    print(table)

# **------------------------------- Product Info -----------------------------**


### **----------------- Created Table product_info via combining Tables 'products' & 'categories'-----------------------**

In [ ]:
product_info = pd.read_sql(
    """
    WITH product_cte AS (
        SELECT
           c.category_id,
           c.category_name,
           c.department,
           p.product_id,
           p.product_name,
           p.brand AS product_brand,
           p.cost_price,
           p.unit_price,
           p.stock_quantity,
           p.is_active,
           p.created_date,
           p.avg_rating
        FROM products p
        JOIN categories c
        ON p.category_id = c.category_id
    )



    SELECT * FROM product_cte
    """,con
)

In [ ]:
product_info

# **------------------------------- OBT SALES -----------------------------**


#### **----------------- Created Table sales_info via combining Tables 'customers' & 'orders' & 'orders_items' & 'payemnts'-----------------------**

In [19]:
sales_info = pd.read_sql(
    """
    WITH sales_cte AS (
        SELECT 
           c.customer_id,
           o.order_id,
           oi.product_id,
           o.order_date,
           o.order_status,
           o.order_channel,
           o.shipping_state,
           o.shipping_city,
           o.shipping_fee,
           o.order_total,
           o.order_subtotal, 
           oi.quantity,
           oi.unit_price,
           oi.discount_percent,
           oi.line_total,
           ps.payment_method,
           ps.payment_status,
           ps.amount,
           ps.transaction_fee
           
           
        FROM customers c
        JOIN orders o
        ON c.customer_id = o.customer_id
        JOIN order_items oi
        ON o.order_id = oi.order_id
        JOIN payments ps
        ON oi.order_id = ps.order_id
    )

    SELECT * FROM sales_cte WHERE payment_status = 'Completed'

    """,con
)

In [20]:
sales_info

,customer_id,order_id,product_id,order_date,order_status,order_channel,shipping_state,shipping_city,shipping_fee,order_total,order_subtotal,quantity,unit_price,discount_percent,line_total,payment_method,payment_status,amount,transaction_fee
0,100438,700000,5276,2025-11-09 16:03:23,Delivered,Website,Delhi,Delhi,0.0,5729.43,5729.43,2,463.18,0,926.36,Debit Card,Completed,5729.43,103.13
1,100438,700000,5312,2025-11-09 16:03:23,Delivered,Website,Delhi,Delhi,0.0,5729.43,5729.43,2,2527.93,5,4803.07,Debit Card,Completed,5729.43,103.13
2,100568,700001,5259,2024-01-26 17:25:29,Delivered,Website,Tamil Nadu,Chennai,0.0,11589.47,11589.47,3,2094.33,0,6282.99,Credit Card,Completed,11589.47,208.61
3,100568,700001,5312,2024-01-26 17:25:29,Delivered,Website,Tamil Nadu,Chennai,0.0,11589.47,11589.47,1,2527.93,10,2275.14,Credit Card,Completed,11589.47,208.61
4,100568,700001,5483,2024-01-26 17:25:29,Delivered,Website,Tamil Nadu,Chennai,0.0,11589.47,11589.47,4,797.72,5,3031.34,Credit Card,Completed,11589.47,208.61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28531,102907,711998,5483,2023-12-22 16:59:35,Delivered,Website,Tamil Nadu,Chennai,0.0,222069.02,222069.02,3,797.72,10,2153.84,Debit Card,Completed,222069.02,3997.24
28532,101018,711999,5042,2025-08-22 22:37:10,Delivered,Mobile App,Gujarat,Ahmedabad,0.0,81334.36,81334.36,1,66334.75,0,66334.75,Debit Card,Completed,81334.36,1464.02
28533,101018,711999,5276,2025-08-22 22:37:10,Delivered,Mobile App,Gujarat,Ahmedabad,0.0,81334.36,81334.36,4,463.18,0,1852.72,Debit Card,Completed,81334.36,1464.02
28534,101018,711999,5312,2025-08-22 22:37:10,Delivered,Mobile App,Gujarat,Ahmedabad,0.0,81334.36,81334.36,5,2527.93,0,12639.65,Debit Card,Completed,81334.36,1464.02


## **---------------------------Store Table 'sales_info' In SqLite---------------------------**

In [8]:
cursor = con.cursor()

In [16]:
cursor.execute("""
    
    CREATE TABLE sale_info (
        customer_id INT,
        order_id INT,
        product_id INT,
        order_date DATE,
        order_status VARCHAR(50),
        order_channel VARCHAR(40),
        shipping_state VARCHAR(40),
        shipping_city VARCHAR(40),
        shipping_fee FLOAT,
        order_total DECIMAL(12,2),
        order_subtotal DECIMAL(12,2),
        order_quantity INT,
        unit_price DECIMAL(12,2),
        discount_percent FLOAT,
        line_total DECIMAL(12,2),
        payment_method VARCHAR(50),
        payment_status VARCHAR(50),
        total_amount DECIMAL(12,2),
        transaction_fee FLOAT
);
""")



OperationalError: table sale_info already exists

In [21]:
sales_info.to_sql('sale_info', con, if_exists = 'replace', index = False)

28536

In [30]:
obt_sale_info = pd.read_sql("SELECT * FROM sale_info",con)

In [16]:
print(con)

## **---------------------------Store Table 'customer_info' In SqLite---------------------------**

In [ ]:
cursor.execute(
    """
    CREATE TABLE customer_info (
        customer_id INT,
        customer_name VARCHAR(50),
        email VARCHAR(200),
        phone VARCHAR(50),
        gender VARCHAR(30),
        date_of_birth DATE,
        city VARCHAR(50),
        state VARCHAR(50),
        country VARCHAR(50),
        signup_date DATE,
        cuatomer_segment VARCHAR(50),
        is_subscribed_newsletter INT
    
    )
    """
)

In [ ]:
customers.to_sql('customer_info', con, if_exists = 'replace', index = False)

In [28]:
customer_info = pd.read_sql("SELECT * FROM customer_info",con)

## **---------------------------Store Table 'product_info' In SqLite---------------------------**

In [ ]:
cursor.execute(
    """
    CREATE TABLE products_info (
           category_id INT ,
           category_name VARCHAR(50),
           department VARCHAR(50),
           product_id INT,
           product_name VARCHAR(100),
           product_brand VARCHAR(50),
           cost_price DECIMAL(12,2),
           unit_price DECIMAL(12,2),
           stock_quantity INT,
           is_active INT,
           created_date DATE,
           avg_rating FLOAT
        
    )
    """
)

In [26]:
products_info = pd.read_sql("SELECT * FROM products_info",con)

In [ ]:
product_info.to_sql('products_info', con, if_exists = 'replace', index = False)

In [27]:
products_info.to_csv("D:\project_pandas\project_01\cleaned_data_for_analytics\product_with_category.csv", index = False)

In [29]:
customer_info.to_csv("D:\project_pandas\project_01\cleaned_data_for_analytics\customer.csv", index = False)

In [31]:
obt_sale_info.to_csv("D:\project_pandas\project_01\cleaned_data_for_analytics\obt_sale_info.csv", index = False)